# 🌾 GraminRoute — ML Training Notebook
**Version 2.0 · Jangaon District, Telangana**

This notebook trains all three models used in the GraminRoute inference pipeline and saves them to `backend/models/`. The FastAPI server loads these on startup without retraining.

| # | Stage | Model | Output |
|---|-------|-------|--------|
| 1 | Feature Engineering | Festival Calendar | 9-feature vector |
| 2 | Per-shop Risk | **XGBoost Classifier** | Risk score [0–1] |
| 3 | Spatial Propagation | **SpatialGNN (GATv2)** | Graph-aware risk |
| 4 | Distributor Ranking | **XGBoost Multi-class** | Ranked recommendations |

---


In [ ]:
# ── Colab Setup (skip if running locally) ─────────────────────────────────────
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import os
    # Clone repo and move into notebook/ so all relative paths resolve correctly
    if not os.path.exists('/content/Gramin-Route'):
        !git clone https://github.com/Atharva-sp21/Gramin-Route.git /content/Gramin-Route
    os.chdir('/content/Gramin-Route/notebook')

    # Install backend dependencies
    !pip install -q torch torchvision torch-geometric xgboost scikit-learn shap networkx nbformat fastapi uvicorn

    print("✅  Colab environment ready")
    print(f"   Working dir: {os.getcwd()}")
else:
    print("Running locally — no Colab setup needed")


## 0 · Setup & Imports

In [ ]:
import sys, os, warnings, pickle
from pathlib import Path
from datetime import date, timedelta

# ── Path setup ─────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
BACKEND_DIR  = REPO_ROOT / "backend"
DATA_DIR     = REPO_ROOT / "data"
MODELS_DIR   = BACKEND_DIR / "models"

sys.path.insert(0, str(BACKEND_DIR))
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Core imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
RISK_CMAP = plt.cm.RdYlGn_r   # green=safe → red=high risk

print("✅  Imports ready")
print(f"   Data    : {DATA_DIR}")
print(f"   Models  : {MODELS_DIR}")


## 1 · Data Loading & Exploratory Analysis

The raw dataset covers **100 kirana (small grocery) shops** across Jangaon District. We have 12 columns, but the key modelling target is `Late_delivery_risk`.


In [ ]:
# Real supply chain data — 500 Jangaon District shops
# Derived from DataCo Global Supply Chain dataset (180k real orders)
# aggregated to shop level, remapped to Jangaon lat/lon, Indian product names.

df = pd.read_csv(DATA_DIR / "jangaon_shops.csv")
print(f"Shape: {df.shape}")
df.head(8)


In [ ]:
df.describe().round(2)


### 1.1 Feature Distributions

In [ ]:
numeric_cols = ['Stock', 'Sales', 'Days', 'Profit_Margin', 'Shelf_Life', 'Credit_Score']
col_labels   = ['Stock (units)', 'Daily Sales', 'Lead Time (days)',
                'Profit Margin (%)', 'Shelf Life (days)', 'Credit Score']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

colors = sns.color_palette("husl", 6)
for i, (col, label) in enumerate(zip(numeric_cols, col_labels)):
    ax = axes[i]
    ax.hist(df[col], bins=25, color=colors[i], edgecolor='white', alpha=0.85)
    ax.set_title(label)
    ax.set_ylabel('Count')
    mean_val = df[col].mean()
    ax.axvline(mean_val, color='black', linestyle='--', linewidth=1.2,
               label=f'Mean: {mean_val:.1f}')
    ax.legend(fontsize=9)

fig.suptitle('Feature Distributions — 500 Jangaon District Shops (DataCo-derived)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_distributions.png', bbox_inches='tight')
plt.show()
print("500 real shop records from DataCo global supply chain data")


### 1.2 Correlation Heatmap

In [ ]:
corr_cols = numeric_cols + ['Festival_Flag', 'Late_delivery_risk']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, ax=ax,
            annot_kws={'size': 9},
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — Jangaon Shops', fontsize=14,
             fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_correlation.png', bbox_inches='tight')
plt.show()


### 1.3 Risk Label Distribution

`Late_delivery_risk` has discrete values: 0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0 — we binarise at **0.5** for XGBoost training.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: raw continuous distribution from DataCo
axes[0].hist(df['Late_delivery_risk'], bins=15, color='#e67e22',
             edgecolor='white', alpha=0.85)
axes[0].set_title('Late Delivery Risk — Raw (DataCo real data)')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Shop Count')

# Right: binarised
binary_labels = (df['Late_delivery_risk'] >= 0.5).astype(int)
pie_data = binary_labels.value_counts()
wedge_colors = ['#27ae60', '#e74c3c']
axes[1].pie(pie_data.values, labels=['Low Risk (0)', 'High Risk (1)'],
            colors=wedge_colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 11}, pctdistance=0.75,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Binarised Labels (threshold >= 0.5)')

plt.suptitle('Late Delivery Risk — 500 Jangaon Shops', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_risk_labels.png', bbox_inches='tight')
plt.show()

high_risk_pct = binary_labels.mean() * 100
print(f"High-risk shops: {binary_labels.sum()} / {len(binary_labels)} ({high_risk_pct:.1f}%)")
print("Source: DataCo real delivery outcome data (Late_delivery_risk aggregated per location)")


### 1.4 Village Map — Shops by Location & Risk

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

sc = ax.scatter(df['Longitude'], df['Latitude'],
                c=df['Late_delivery_risk'], cmap=RISK_CMAP,
                s=60, edgecolors='white', linewidths=0.4,
                alpha=0.80, zorder=3)

# Annotate top 10 highest-risk shops
high_risk = df.nlargest(10, 'Late_delivery_risk')
for _, row in high_risk.iterrows():
    ax.annotate(row['Village_Name'], (row['Longitude'], row['Latitude']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points',
                color='#c0392b')

cbar = plt.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Late Delivery Risk', fontsize=11)
ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Jangaon District — 500 Shop Locations Coloured by Risk\n'
             '(Real delivery risk from DataCo supply chain data)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'eda_village_map.png', bbox_inches='tight')
plt.show()


## 2 · Feature Engineering — Festival Context

The raw dataset has a binary `Festival_Flag`. We replace it with **3 continuous features** derived from the Indian festival calendar, giving the model richer temporal context.

| Old | New |
|-----|-----|
| `Festival_Flag` (0/1) | `days_to_festival` (0–30, normalised) |
| — | `spike_factor` (1.0–2.8 demand multiplier) |
| — | `product_affinity` (1.0 if this product is affected) |


In [ ]:
from ml.festival_calendar import get_festival_context, FESTIVAL_CALENDAR

# Show the calendar
print("📅 Festival Calendar — Jangaon District")
print("=" * 60)
for name, info in FESTIVAL_CALENDAR.items():
    print(f"  {name:<12} | Month {info['month']:>2} Day {info['day']:>2} | "
          f"Spike ×{info['spike_factor']:.1f} | Prep {info['prep_days']} days")
    print(f"             | Products: {', '.join(info['products'])}")


In [ ]:
# Product names are already in the CSV (one per shop)
# Derive festival context for each shop based on its product

TODAY = date.today()

festival_feats = df['Product_Name'].apply(
    lambda p: get_festival_context(p, TODAY)
)

df['days_to_festival_raw'] = [f['days_to_festival'] for f in festival_feats]
df['spike_factor']         = [f['spike_factor']     for f in festival_feats]
df['product_affinity']     = [f['product_affinity'] for f in festival_feats]
df['festival_name']        = [f['festival_name']    for f in festival_feats]
df['in_prep_window']       = [f['in_prep_window']   for f in festival_feats]

df['days_to_festival_norm'] = (df['days_to_festival_raw'] / 30.0).clip(0, 1)
df['spike_factor_norm']     = df['spike_factor'] / 3.0

print("Festival features added to all 500 shops:")
print(df[['Village_Name', 'Product_Name', 'festival_name',
          'days_to_festival_raw', 'spike_factor', 'product_affinity']].head(10).to_string(index=False))


### 2.1 Festival Feature Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Spike factor by festival
festival_spike = {k: v['spike_factor'] for k, v in FESTIVAL_CALENDAR.items()}
bars = axes[0].barh(list(festival_spike.keys()), list(festival_spike.values()),
                    color=sns.color_palette('Oranges_r', len(festival_spike)),
                    edgecolor='white')
axes[0].set_xlabel('Demand Spike Factor (×baseline)')
axes[0].set_title('Demand Spike by Festival')
for bar, val in zip(bars, festival_spike.values()):
    axes[0].text(val + 0.02, bar.get_y() + bar.get_height()/2,
                 f'×{val}', va='center', fontsize=10, fontweight='bold')
axes[0].set_xlim(0, 3.5)
axes[0].axvline(1.0, color='grey', linestyle='--', linewidth=1, label='Baseline')
axes[0].legend()

# 2. Product affinity distribution
affinity_counts = df['product_affinity'].value_counts()
axes[1].bar(['Not Affected\n(affinity=0)', 'Festival Product\n(affinity=1)'],
            [affinity_counts.get(0.0, 0), affinity_counts.get(1.0, 0)],
            color=['#3498db', '#e67e22'], edgecolor='white', width=0.5)
axes[1].set_ylabel('Number of Shops')
axes[1].set_title('Product Affinity Distribution\n(all 100 shops)')
for i, v in enumerate([affinity_counts.get(0.0, 0), affinity_counts.get(1.0, 0)]):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontsize=11, fontweight='bold')

# 3. Days to festival histogram
axes[2].hist(df['days_to_festival_raw'], bins=15, color='#9b59b6',
             edgecolor='white', alpha=0.85)
axes[2].set_xlabel('Days to Next Festival')
axes[2].set_ylabel('Count')
axes[2].set_title('Days to Next Festival\n(per product)')
axes[2].axvline(df['days_to_festival_raw'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f"Mean: {df['days_to_festival_raw'].mean():.0f}d")
axes[2].legend()

plt.suptitle('Festival Feature Engineering — Before vs After', fontsize=14,
             fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'feat_eng_festival.png', bbox_inches='tight')
plt.show()


In [ ]:
# Scatter: festival proximity vs late delivery risk
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Old: binary Festival_Flag vs risk
jitter_x = df['Festival_Flag'] + np.random.normal(0, 0.02, len(df))
axes[0].scatter(jitter_x, df['Late_delivery_risk'],
                alpha=0.5, c=df['Late_delivery_risk'], cmap=RISK_CMAP,
                edgecolors='white', linewidths=0.4, s=60)
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Festival', 'Festival'])
axes[0].set_ylabel('Late Delivery Risk'); axes[0].set_title('Old: Binary Festival Flag')

# New: days_to_festival (continuous) vs risk
sc = axes[1].scatter(df['days_to_festival_raw'], df['Late_delivery_risk'],
                     alpha=0.65, c=df['spike_factor'], cmap='Oranges',
                     edgecolors='white', linewidths=0.4, s=80)
cb = plt.colorbar(sc, ax=axes[1], shrink=0.9)
cb.set_label('Spike Factor')
axes[1].set_xlabel('Days to Next Festival')
axes[1].set_ylabel('Late Delivery Risk')
axes[1].set_title('New: Continuous Festival Features')

plt.suptitle('Festival Feature: Before vs After', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'feat_eng_comparison.png', bbox_inches='tight')
plt.show()


## 3 · XGBoost Risk Model

**Architecture:** Binary XGBoost classifier on 9 features.

**Training strategy:** The 100 CSV rows are augmented with 5,000 synthetic samples generated by rule-based labelling (knowledge distillation). This gives the model enough variation to generalise.

```
FEATURE_NAMES = [
    stock_ratio, sales_ratio, lead_time_ratio, margin_ratio,
    shelf_life_ratio, credit_ratio,          ← from CSV
    days_to_festival, spike_factor, product_affinity   ← from Festival Calendar
]
```


In [ ]:
FEATURE_NAMES = [
    "stock_ratio", "sales_ratio", "lead_time_ratio",
    "margin_ratio", "shelf_life_ratio", "credit_ratio",
    "days_to_festival", "spike_factor", "product_affinity",
]

# ── Build REAL feature matrix from jangaon_shops.csv (DataCo-derived) ─────────
X_real = np.column_stack([
    df['Stock'].values         / 200.0,
    df['Sales'].values         / 50.0,
    df['Days'].values          / 7.0,
    df['Profit_Margin'].values  / 60.0,
    df['Shelf_Life'].values    / 365.0,
    df['Credit_Score'].values  / 900.0,
    df['days_to_festival_norm'].values,
    df['spike_factor_norm'].values,
    df['product_affinity'].values,
]).astype(np.float32)

y_real = (df['Late_delivery_risk'].values >= 0.5).astype(int)
print(f"Real data shape: {X_real.shape}  |  High-risk: {y_real.sum()} / {len(y_real)}")
print("Source: 500 real Jangaon shops (DataCo supply chain data)")

# ── Augment with 2,000 synthetic samples (much smaller — real data dominates) ──
np.random.seed(42)
N = 2000

s_stock    = np.random.uniform(0, 1, N)
s_sales    = np.random.uniform(0, 0.5, N)
s_lead     = np.random.uniform(0, 1, N)
s_margin   = np.random.uniform(0, 1, N)
s_shelf    = np.random.uniform(0, 1, N)
s_credit   = np.random.uniform(0.4, 1.0, N)
s_dtf      = np.random.uniform(0, 1, N)
s_spike    = np.random.uniform(0.33, 1.0, N)
s_affinity = np.random.randint(0, 2, N).astype(float)

X_synth = np.column_stack([s_stock, s_sales, s_lead, s_margin,
                            s_shelf, s_credit, s_dtf, s_spike, s_affinity])

effective_sales = s_sales * 50 * np.where(s_affinity > 0.5, s_spike * 3.0, 1.0)
days_out = np.where(effective_sales > 0, (s_stock * 200) / (effective_sales + 1e-6), 30.0)
y_synth = (
    (days_out < s_lead * 7) |
    ((s_affinity > 0.5) & (days_out < s_dtf * 30)) |
    (s_stock < 0.10)
).astype(int)

# Inject realism
X_synth += np.random.normal(0, 0.05, X_synth.shape)
X_synth = np.clip(X_synth, 0, 1)
noise_idx = np.random.choice(N, size=int(0.10 * N), replace=False)
y_synth[noise_idx] = 1 - y_synth[noise_idx]
borderline = np.abs(days_out - s_lead * 7) < 1.5
y_synth[borderline] = np.random.randint(0, 2, borderline.sum())

# Combine — real data weighted 1x, synthetic 0.4x effective via smaller N
X_all = np.vstack([X_real, X_synth])
y_all = np.concatenate([y_real, y_synth])

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"\nCombined shape: {X_all.shape}")
print(f"  Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"  High-risk in train: {y_train.mean()*100:.1f}%")
print(f"\nReal data makes up {len(X_real)/len(X_all)*100:.0f}% of training set")


In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
xgb_risk = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
    early_stopping_rounds=20,
)
xgb_risk.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

# ── Training loss curve ───────────────────────────────────────────────────────
results = xgb_risk.evals_result()
train_loss = results['validation_0']['logloss']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_loss, color='#e74c3c', linewidth=2, label='Validation log-loss')
ax.axhline(min(train_loss), color='grey', linestyle='--', linewidth=1,
           label=f'Best: {min(train_loss):.4f} @ iter {train_loss.index(min(train_loss))}')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('Log-loss')
ax.set_title('XGBoost Risk Model — Training Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_training_curve.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
y_prob = xgb_risk.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
axes[0].plot(fpr, tpr, color='#e74c3c', linewidth=2.5,
             label=f'AUC = {roc_auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#e74c3c')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — XGBoost Risk Model', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=12)
axes[0].set_xlim([0, 1]); axes[0].set_ylim([0, 1.01])

# Confusion matrix
y_pred = xgb_risk.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Low Risk', 'High Risk'])
disp.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix', fontweight='bold')
axes[1].grid(False)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_roc_cm.png', bbox_inches='tight')
plt.show()

print(f"\nAUC: {roc_auc:.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))


### 3.1 SHAP Feature Importance

SHAP (SHapley Additive exPlanations) shows *exactly how much* each feature pushes the risk score up or down for each individual prediction.

In [ ]:
# ── SHAP Analysis ─────────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(xgb_risk)
shap_values = explainer.shap_values(X_test[:500])   # sample for speed

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Beeswarm plot (impact distribution per feature)
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test[:500],
                  feature_names=FEATURE_NAMES,
                  plot_type='dot', show=False)
axes[0].set_title('SHAP Beeswarm — Impact Distribution', fontweight='bold', pad=10)

# Bar plot (mean absolute SHAP = overall importance)
plt.sca(axes[1])
shap.summary_plot(shap_values, X_test[:500],
                  feature_names=FEATURE_NAMES,
                  plot_type='bar', show=False)
axes[1].set_title('SHAP Bar — Mean |Impact| per Feature', fontweight='bold', pad=10)

plt.savefig(MODELS_DIR / 'risk_shap.png', bbox_inches='tight', dpi=120)
plt.show()


In [ ]:
# ── SHAP Waterfall — single shop explanation ─────────────────────────────────
shop_idx = np.argmax(y_prob)   # most-at-risk test sample

fig, ax = plt.subplots(figsize=(10, 5))
shap_ex = shap.Explanation(
    values         = shap_values[shop_idx],
    base_values    = explainer.expected_value,
    data           = X_test[shop_idx],
    feature_names  = FEATURE_NAMES,
)
shap.waterfall_plot(shap_ex, show=False)
plt.title(f'SHAP Waterfall — Single High-Risk Shop\n'
          f'Predicted Risk: {y_prob[shop_idx]:.2%}',
          fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'risk_shap_waterfall.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
risk_model_path = MODELS_DIR / "xgb_risk_model.pkl"
with open(risk_model_path, "wb") as f:
    pickle.dump(xgb_risk, f)
print(f"✅  XGBoost Risk Model saved → {risk_model_path}")


## 4 · Village Road Graph Construction

Before training the GNN, we build a geographic graph from the 100 shop coordinates:
- **Nodes** = shops (features = 10-dim vector)
- **Edges** = connections between shops within 15 km
- **Edge weights** = road type (highway=1.0, state road=0.6, rural=0.3)


In [ ]:
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
from sklearn.neighbors import NearestNeighbors

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

G = nx.Graph()
lats  = df['Latitude'].values
lons  = df['Longitude'].values
risks = (df['Late_delivery_risk'].values >= 0.5).astype(float)

for i in range(len(df)):
    G.add_node(i, lat=lats[i], lon=lons[i], risk=risks[i],
               village=df['Village_Name'].iloc[i])

# K-nearest neighbours (k=5) — realistic sparse road network
K = 5
coords_rad = np.radians([[lats[i], lons[i]] for i in range(len(df))])
nbrs = NearestNeighbors(n_neighbors=K+1, metric='haversine').fit(coords_rad)
distances, indices = nbrs.kneighbors(coords_rad)

for i in range(len(df)):
    for j_pos in range(1, K+1):
        j    = indices[i][j_pos]
        dist = distances[i][j_pos] * 6371.0
        if not G.has_edge(i, j):
            if dist < 3:
                wt, rt = 1.0, 'highway'
            elif dist < 8:
                wt, rt = 0.6, 'state'
            else:
                wt, rt = 0.3, 'rural'
            G.add_edge(i, j, weight=wt, road_type=rt, dist_km=round(dist, 2))

from collections import Counter
road_types = Counter(nx.get_edge_attributes(G, 'road_type').values())
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.1f}")
print(f"Connected components: {nx.number_connected_components(G)}")
print(f"Road types: {dict(road_types)}")


In [ ]:
# ── Visualise the village road graph ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 9))

pos = {i: (lons[i], lats[i]) for i in range(len(df))}

# Edge colours by road type
edge_colors, edge_widths, edge_alphas = [], [], []
for u, v, data in G.edges(data=True):
    rt = data['road_type']
    if rt == 'highway':
        edge_colors.append('#2c3e50'); edge_widths.append(2.2); edge_alphas.append(0.8)
    elif rt == 'state':
        edge_colors.append('#7f8c8d'); edge_widths.append(1.4); edge_alphas.append(0.5)
    else:
        edge_colors.append('#bdc3c7'); edge_widths.append(0.8); edge_alphas.append(0.35)

# Node colours by risk
node_colors = [RISK_CMAP(risks[i]) for i in range(len(df))]
node_sizes  = [150 + 300 * risks[i] for i in range(len(df))]

nx.draw_networkx_edges(G, pos, ax=ax,
                       edge_color=edge_colors, width=edge_widths, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax,
                       node_color=node_colors, node_size=node_sizes,
                       edgecolors='white', linewidths=0.5)

# Label high-risk nodes
high_risk_nodes = {i: df['Village_Name'].iloc[i][:8]
                   for i in range(len(df)) if risks[i] == 1}
nx.draw_networkx_labels(G, pos, labels=high_risk_nodes, ax=ax,
                        font_size=7, font_color='#c0392b')

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='#2c3e50', linewidth=2, label='Highway (w=1.0)'),
    Line2D([0], [0], color='#7f8c8d', linewidth=1.4, label='State Road (w=0.6)'),
    Line2D([0], [0], color='#bdc3c7', linewidth=0.8, label='Rural Road (w=0.3)'),
    plt.scatter([], [], c='#e74c3c', s=80, label='High Risk Shop', edgecolors='white'),
    plt.scatter([], [], c='#27ae60', s=50, label='Low Risk Shop', edgecolors='white'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, framealpha=0.9)
ax.set_title('Jangaon Village Road Network\nNode size & colour = risk level',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'village_graph.png', bbox_inches='tight', dpi=120)
plt.show()


## 5 · SpatialGNN Training (GATv2)

The SpatialGNN takes the XGBoost risk scores as input and propagates information across the road network using **Graph Attention v2 (GATv2)** — a more expressive variant that conditions attention on both source and destination node features.

**Architecture:**
```
Input (10 features): 9 shop features + XGBoost risk score
→ Linear projection → 64 hidden dim
→ GATv2Conv (4 attention heads, edge_dim=1 for road weight)
→ GATv2Conv (1 head, output)
→ Skip connection (0.1 weight) → Sigmoid output
```


In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch_geometric.data import Data
    from torch_geometric.nn import GATv2Conv
    TORCH_OK = True
    print(f"✅  PyTorch {torch.__version__} | PyG ready")
except ImportError as e:
    TORCH_OK = False
    print(f"⚠️  PyTorch/PyG not found: {e}")
    print("    Install with: pip install torch torch-geometric")
    print("    Skipping GNN sections — XGBoost risk scores will be used directly.")


In [ ]:
if TORCH_OK:
    from ml.model_def import SpatialGNN

    # ── Build PyTorch Geometric Data from our NetworkX graph ─────────────────
    # Node features: 9-dim normalised + XGBoost risk as 10th dim
    X_node_np = np.column_stack([
        df['Stock'].values         / 200.0,
        df['Sales'].values         / 50.0,
        df['Days'].values          / 7.0,
        df['Profit_Margin'].values  / 60.0,
        df['Shelf_Life'].values    / 365.0,
        df['Credit_Score'].values  / 900.0,
        df['days_to_festival_norm'].values,
        df['spike_factor_norm'].values,
        df['product_affinity'].values,
    ]).astype(np.float32)

    # Get XGBoost risk scores for all shops
    xgb_risk_scores = xgb_risk.predict_proba(X_node_np)[:, 1].astype(np.float32)
    X_10 = np.column_stack([X_node_np, xgb_risk_scores])    # shape: (100, 10)

    node_x     = torch.tensor(X_10, dtype=torch.float32)
    node_y     = torch.tensor(risks, dtype=torch.float32).unsqueeze(1)

    # Edge index and edge attr from NetworkX graph
    edge_list, edge_weights = [], []
    for u, v, data in G.edges(data=True):
        edge_list.append([u, v])
        edge_list.append([v, u])   # undirected → bidirectional
        edge_weights.extend([data['weight'], data['weight']])

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    edge_attr  = torch.tensor(edge_weights, dtype=torch.float32).unsqueeze(1)

    pyg_data = Data(x=node_x, edge_index=edge_index, edge_attr=edge_attr, y=node_y)
    print(f"PyG Data: {pyg_data}")
    print(f"  Nodes: {pyg_data.num_nodes}, Edges: {pyg_data.num_edges}")


In [ ]:
if TORCH_OK:
    # ── Training loop ─────────────────────────────────────────────────────────
    model     = SpatialGNN(in_dim=10, hidden_dim=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    criterion = nn.BCELoss()

    # Train/val split by node index
    n_nodes   = pyg_data.num_nodes
    idx_all   = torch.randperm(n_nodes)
    n_train   = int(0.8 * n_nodes)
    train_idx = idx_all[:n_train]
    val_idx   = idx_all[n_train:]

    train_losses, val_losses, val_accs = [], [], []

    model.train()
    for epoch in range(1, 201):
        optimizer.zero_grad()
        out = model(pyg_data)

        train_loss = criterion(out[train_idx], pyg_data.y[train_idx])
        train_loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_out   = model(pyg_data)
            val_loss  = criterion(val_out[val_idx], pyg_data.y[val_idx]).item()
            val_pred  = (val_out[val_idx] > 0.5).float()
            val_acc   = (val_pred == pyg_data.y[val_idx]).float().mean().item()
        model.train()

        train_losses.append(train_loss.item())
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if epoch % 40 == 0:
            print(f"Epoch {epoch:>3} | Train Loss: {train_loss.item():.4f} "
                  f"| Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.3f}")

    print(f"\nBest val accuracy: {max(val_accs):.3f} @ epoch {val_accs.index(max(val_accs)) + 1}")


In [ ]:
if TORCH_OK:
    # ── Training curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    epochs = range(1, 201)
    axes[0].plot(epochs, train_losses, color='#e74c3c', linewidth=1.8, label='Train Loss')
    axes[0].plot(epochs, val_losses,   color='#3498db', linewidth=1.8,
                 linestyle='--', label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
    axes[0].set_title('SpatialGNN — Training & Validation Loss', fontweight='bold')
    axes[0].legend()

    axes[1].plot(epochs, val_accs, color='#27ae60', linewidth=1.8)
    axes[1].axhline(max(val_accs), color='grey', linestyle='--', linewidth=1,
                    label=f'Peak: {max(val_accs):.3f}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('SpatialGNN — Validation Accuracy', fontweight='bold')
    axes[1].set_ylim([0, 1.05]); axes[1].legend()

    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gnn_training_curves.png', bbox_inches='tight')
    plt.show()


In [ ]:
if TORCH_OK:
    # ── Compare XGBoost risk vs GNN spatial risk ──────────────────────────────
    model.eval()
    with torch.no_grad():
        gnn_risk_scores = model(pyg_data).squeeze().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 1. XGBoost risk distribution
    axes[0].hist(xgb_risk_scores, bins=20, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[0].set_title('XGBoost Risk Scores\n(per-shop, no graph context)')
    axes[0].set_xlabel('Risk Score'); axes[0].set_ylabel('Count')

    # 2. GNN spatial risk distribution
    axes[1].hist(gnn_risk_scores, bins=20, color='#3498db', edgecolor='white', alpha=0.8)
    axes[1].set_title('SpatialGNN Risk Scores\n(road-network aware)')
    axes[1].set_xlabel('Risk Score')

    # 3. Scatter: XGBoost vs GNN (colour = ground truth)
    sc = axes[2].scatter(xgb_risk_scores, gnn_risk_scores,
                         c=risks, cmap='RdYlGn_r',
                         s=60, edgecolors='white', linewidths=0.5, alpha=0.8)
    axes[2].plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
    plt.colorbar(sc, ax=axes[2], label='Ground Truth Risk')
    axes[2].set_xlabel('XGBoost Risk Score')
    axes[2].set_ylabel('GNN Spatial Risk Score')
    axes[2].set_title('XGBoost vs SpatialGNN\nPoints above diagonal = GNN elevated risk')

    plt.suptitle('Risk Score Comparison: Individual vs Spatial', fontsize=13,
                 fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(MODELS_DIR / 'gnn_risk_comparison.png', bbox_inches='tight')
    plt.show()


In [ ]:
if TORCH_OK:
    # ── Save GNN weights ──────────────────────────────────────────────────────
    gnn_path = MODELS_DIR / "spatial_gnn.pth"
    torch.save(model.state_dict(), gnn_path)
    print(f"✅  SpatialGNN weights saved → {gnn_path}")
else:
    print("⚠️  Skipped GNN training. XGBoost risk scores will be used directly by the API.")


## 6 · XGBoost Distributor Recommender

**Framing:** Contextual Bandit — the *context* is the retailer's current state; the *arms* are the 3 distributors; the *reward* is delivery success × cost efficiency.

We approximate the optimal policy with XGBoost multi-class classification trained on synthetic context–action–reward data. As real delivery data accumulates, this can be swapped for LinUCB or Thompson Sampling.

| Class | Distributor | Cost | ETA | Reliability |
|-------|-------------|------|-----|-------------|
| 0 | FastTrack Logistics | ₹100 | 4 hrs | 99% |
| 1 | GraminRoute Hub | ₹75 | 12 hrs | 95% |
| 2 | Budget Movers | ₹60 | 24 hrs | 85% |

**Context features:** `[spatial_risk, days_stockout/30, days_festival/30, credit/900, qty/100]`


In [ ]:
np.random.seed(42)
N_REC = 4000

r_risk     = np.random.uniform(0, 1, N_REC)
r_stockout = np.random.uniform(0.1, 1.0, N_REC)     # days_until_stockout / 30
r_festival = np.random.uniform(0, 1, N_REC)          # days_to_festival / 30
r_credit   = np.random.uniform(0.4, 1.0, N_REC)
r_qty      = np.random.uniform(0, 1, N_REC)          # qty / 100

X_rec = np.column_stack([r_risk, r_stockout, r_festival, r_credit, r_qty])

# Rule: high risk or imminent stockout → FastTrack (0)
#       moderate risk → GraminRoute Hub (1)
#       low risk → Budget Movers (2)
y_rec = np.where(
    r_risk > 0.7, 0,
    np.where(r_stockout < 0.17, 0,      # < 5 days stockout → FastTrack
    np.where(r_risk > 0.4, 1, 2))
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rec, y_rec, test_size=0.2, random_state=42, stratify=y_rec
)

xgb_rec = xgb.XGBClassifier(
    n_estimators=150, max_depth=5, learning_rate=0.1,
    eval_metric="mlogloss", random_state=42,
)
xgb_rec.fit(X_train_r, y_train_r)

y_pred_r = xgb_rec.predict(X_test_r)
print("Recommender Classification Report:")
print(classification_report(y_test_r, y_pred_r,
                            target_names=['FastTrack', 'GraminRoute Hub', 'Budget Movers']))


In [ ]:
# ── Confidence distributions ─────────────────────────────────────────────────
probas = xgb_rec.predict_proba(X_test_r)

DIST_NAMES  = ['FastTrack Logistics', 'GraminRoute Hub', 'Budget Movers']
DIST_COLORS = ['#e74c3c', '#3498db', '#27ae60']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for i, (name, color) in enumerate(zip(DIST_NAMES, DIST_COLORS)):
    axes[i].hist(probas[:, i], bins=25, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{name}\nConfidence Distribution', fontweight='bold')
    axes[i].set_xlabel('Predicted Probability')
    axes[i].set_ylabel('Count' if i == 0 else '')
    mean_conf = probas[:, i].mean()
    axes[i].axvline(mean_conf, color='black', linestyle='--', linewidth=1.5,
                    label=f'Mean: {mean_conf:.2f}')
    axes[i].legend()

plt.suptitle('Distributor Recommendation — Confidence Distributions\n'
             '(XGBoost Multi-class Classifier)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'recommender_confidence.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Confusion matrix ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
cm_rec = confusion_matrix(y_test_r, y_pred_r)
disp_r = ConfusionMatrixDisplay(cm_rec,
                                display_labels=['FastTrack', 'Hub', 'Budget'])
disp_r.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Recommender — Confusion Matrix', fontweight='bold')
ax.grid(False)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'recommender_cm.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature importance ───────────────────────────────────────────────────────
rec_features = ['spatial_risk', 'days_stockout', 'days_festival',
                'credit_score', 'order_qty']
importances  = xgb_rec.feature_importances_

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(rec_features, importances,
               color=sns.color_palette('Blues_r', len(rec_features)),
               edgecolor='white')
ax.set_xlabel('Feature Importance (F-score)')
ax.set_title('XGBoost Recommender — Feature Importances', fontweight='bold')
for bar, val in zip(bars, importances):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlim(0, max(importances) * 1.25)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'recommender_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Save recommender model ────────────────────────────────────────────────────
rec_path = MODELS_DIR / "xgb_recommender.pkl"
with open(rec_path, "wb") as f:
    pickle.dump(xgb_rec, f)
print(f"✅  XGBoost Recommender saved → {rec_path}")


## 7 · End-to-End Demo — Ramesh's Shop

> **Scenario:** Ramesh runs a kirana shop in Ghanpur village. He has 18 kg of Rice (50kg bags) left, sells ~8 bags/day, has a 3-day lead time, and Diwali is approaching. What should GraminRoute recommend?

We run the same pipeline the FastAPI `/recommend_distributor` endpoint uses.


In [ ]:
from ml.festival_calendar import get_festival_context
from ml.festival_predictor import compute_stockout_forecast

# ── Ramesh's shop parameters ──────────────────────────────────────────────────
RAMESH = dict(
    shop_id       = "GHPR-042",
    village       = "Ghanpur",
    current_stock = 18,
    daily_sales   = 8,
    lead_time_days= 3,
    profit_margin = 22.0,
    shelf_life    = 365,
    credit_score  = 720,
    product_name  = "Rice (50kg)",
)

print("=" * 56)
print("  🏪  RAMESH'S KIRANA SHOP — GraminRoute Pipeline Run")
print("=" * 56)
print(f"  Shop     : {RAMESH['shop_id']} — {RAMESH['village']}")
print(f"  Product  : {RAMESH['product_name']}")
print(f"  Stock    : {RAMESH['current_stock']} units")
print(f"  Sales    : {RAMESH['daily_sales']} units/day")
print(f"  Lead time: {RAMESH['lead_time_days']} days")


In [ ]:
# ── STAGE 1: Feature Engineering ─────────────────────────────────────────────
print("\n📌  STAGE 1 — Feature Engineering")
print("-" * 40)

festival_ctx = get_festival_context(RAMESH['product_name'])
print(f"  Nearest festival : {festival_ctx['festival_name']}")
print(f"  Days away        : {festival_ctx['days_to_festival']}")
print(f"  Spike factor     : ×{festival_ctx['spike_factor']}")
print(f"  Product affected : {'YES ✓' if festival_ctx['product_affinity'] == 1 else 'No'}")
print(f"  In prep window   : {'YES 🔔' if festival_ctx['in_prep_window'] else 'Not yet'}")

features_9 = np.array([
    RAMESH['current_stock']  / 200.0,
    RAMESH['daily_sales']    / 50.0,
    RAMESH['lead_time_days'] / 7.0,
    RAMESH['profit_margin']  / 60.0,
    RAMESH['shelf_life']     / 365.0,
    RAMESH['credit_score']   / 900.0,
    min(festival_ctx['days_to_festival'] / 30.0, 1.0),
    festival_ctx['spike_factor'] / 3.0,
    festival_ctx['product_affinity'],
], dtype=np.float32)

print(f"\n  Feature vector (9-dim):")
for name, val in zip(FEATURE_NAMES, features_9):
    print(f"    {name:<25} {val:.4f}")


In [ ]:
# ── STAGE 2: XGBoost Risk Score ──────────────────────────────────────────────
print("\n📌  STAGE 2 — XGBoost Risk Model")
print("-" * 40)

xgb_risk_score = float(xgb_risk.predict_proba(features_9.reshape(1, -1))[0][1])
status = "🔴 CRITICAL" if xgb_risk_score > 0.7 else "🟡 WARNING" if xgb_risk_score > 0.4 else "🟢 STABLE"

print(f"  XGBoost risk score : {xgb_risk_score:.3f}")
print(f"  Shop status        : {status}")

# Top 3 features driving this prediction
shap_single = explainer.shap_values(features_9.reshape(1, -1))[0]
top3 = sorted(zip(FEATURE_NAMES, shap_single), key=lambda x: abs(x[1]), reverse=True)[:3]
print(f"\n  Top 3 risk drivers (SHAP):")
for name, impact in top3:
    arrow = "↑" if impact > 0 else "↓"
    print(f"    {arrow}  {name:<25} SHAP: {impact:+.4f}")


In [ ]:
# ── STAGE 3: SpatialGNN Spatial Risk ─────────────────────────────────────────
print("\n📌  STAGE 3 — SpatialGNN Spatial Propagation")
print("-" * 40)

if TORCH_OK:
    from ml.model_def import SpatialGNN as _SpatialGNN
    _gnn = _SpatialGNN(in_dim=10, hidden_dim=64)
    _gnn.load_state_dict(torch.load(MODELS_DIR / "spatial_gnn.pth", map_location='cpu'))
    _gnn.eval()

    x_10   = torch.tensor(np.append(features_9, xgb_risk_score), dtype=torch.float32).unsqueeze(0)
    e_idx  = torch.tensor([[0], [0]], dtype=torch.long)   # self-loop
    e_attr = torch.tensor([[1.0]], dtype=torch.float32)

    with torch.no_grad():
        spatial_risk = float(np.clip(_gnn(Data(x=x_10, edge_index=e_idx, edge_attr=e_attr)).item(), 0, 1))

    delta = spatial_risk - xgb_risk_score
    print(f"  XGBoost risk   : {xgb_risk_score:.3f}")
    print(f"  Spatial risk   : {spatial_risk:.3f}  ({'+' if delta >= 0 else ''}{delta:.3f} graph adjustment)")
else:
    spatial_risk = xgb_risk_score
    print(f"  GNN not loaded — using XGBoost risk: {spatial_risk:.3f}")


In [ ]:
# ── STAGE 4: Festival Stockout Forecast ──────────────────────────────────────
print("\n📌  STAGE 4 — Festival Predictor")
print("-" * 40)

forecast = compute_stockout_forecast(
    current_stock  = RAMESH['current_stock'],
    daily_sales    = RAMESH['daily_sales'],
    festival_ctx   = festival_ctx,
    lead_time_days = RAMESH['lead_time_days'],
)

print(f"  Effective demand/day  : {forecast['effective_daily_demand']:.1f} units "
      f"(×{forecast['demand_multiplier']:.1f} spike)")
print(f"  Days until stockout   : {forecast['days_until_stockout']:.1f} days")
print(f"  Festival window       : {forecast.get('festival_window', 'N/A')} days")
print(f"  Recommended order qty : {forecast['recommended_order_qty']} units")
print(f"  Restock urgency       : ⚠️  {forecast['restock_urgency']}")
print(f"  Restock deadline      : {forecast['restock_deadline']}")


In [ ]:
# ── STAGE 5: Distributor Recommendation ──────────────────────────────────────
print("\n📌  STAGE 5 — XGBoost Distributor Recommender")
print("-" * 40)

rec_features = np.array([[
    spatial_risk,
    min(forecast['days_until_stockout'] / 30.0, 1.0),
    min(festival_ctx['days_to_festival']  / 30.0, 1.0),
    RAMESH['credit_score'] / 900.0,
    min(forecast['recommended_order_qty'] / 100.0, 1.0),
]])
rec_proba = xgb_rec.predict_proba(rec_features)[0]

DIST_INFO = [
    {"name": "FastTrack Logistics", "cost": 100, "eta": "4 hrs",  "rel": 0.99, "tier": "PREMIUM"},
    {"name": "GraminRoute Hub",     "cost": 75,  "eta": "12 hrs", "rel": 0.95, "tier": "BALANCED"},
    {"name": "Budget Movers",       "cost": 60,  "eta": "24 hrs", "rel": 0.85, "tier": "ECONOMY"},
]

ranked = sorted(
    [{"distributor": d["name"], "confidence": round(float(rec_proba[i]), 3),
      "cost": d["cost"], "eta": d["eta"], "reliability": d["rel"], "tier": d["tier"]}
     for i, d in enumerate(DIST_INFO)],
    key=lambda x: x["confidence"], reverse=True,
)

print(f"\n  {'Rank':<5} {'Distributor':<22} {'Conf':>6} {'Cost':>6} {'ETA':>8} {'Reliability':>12}")
print(f"  {'─'*5} {'─'*22} {'─'*6} {'─'*6} {'─'*8} {'─'*12}")
medals = ['🥇', '🥈', '🥉']
for i, r in enumerate(ranked):
    print(f"  {medals[i]:<5} {r['distributor']:<22} {r['confidence']:>6.1%} "
          f"  ₹{r['cost']:>3}  {r['eta']:>8}  {r['reliability']:>11.0%}")

print(f"\n  ✅  TOP PICK: {ranked[0]['distributor']} (confidence: {ranked[0]['confidence']:.1%})")
print(f"  💡  REASON : High risk score ({spatial_risk:.2f}) and imminent stockout")


In [ ]:
# ── Summary dashboard ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 6))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# 1. Risk gauge
ax1 = fig.add_subplot(gs[0])
theta = np.linspace(0, np.pi, 100)
for t1, t2, c in [(0, 0.4*np.pi, '#27ae60'),
                   (0.4*np.pi, 0.7*np.pi, '#f39c12'),
                   (0.7*np.pi, np.pi, '#e74c3c')]:
    t_seg = np.linspace(t1, t2, 50)
    ax1.fill_between(np.cos(t_seg), np.sin(t_seg) * 0,
                     np.sin(t_seg), alpha=0.3)
    ax1.plot(np.cos(t_seg), np.sin(t_seg), linewidth=8,
             color=c, solid_capstyle='butt')

needle_angle = np.pi * (1 - spatial_risk)
ax1.annotate('', xy=(0.65 * np.cos(needle_angle), 0.65 * np.sin(needle_angle)),
             xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='black',
             lw=2.5, mutation_scale=18))
ax1.set_xlim(-1.2, 1.2); ax1.set_ylim(-0.2, 1.2)
ax1.axis('off')
ax1.text(0, -0.15, f'Spatial Risk: {spatial_risk:.2f}', ha='center',
         fontsize=12, fontweight='bold')
ax1.text(-1.0, -0.05, 'SAFE', fontsize=9, color='#27ae60')
ax1.text(0.8, -0.05, 'CRITICAL', fontsize=9, color='#e74c3c')
ax1.set_title("Risk Gauge", fontweight='bold', pad=5)

# 2. Stockout timeline
ax2 = fig.add_subplot(gs[1])
days = np.arange(0, 15)
stock_traj = np.maximum(0, RAMESH['current_stock']
                        - days * forecast['effective_daily_demand'])
ax2.fill_between(days, stock_traj, alpha=0.35, color='#3498db')
ax2.plot(days, stock_traj, color='#2980b9', linewidth=2.5)
ax2.axhline(RAMESH['lead_time_days'] * forecast['effective_daily_demand'],
            color='#e74c3c', linestyle='--', linewidth=1.5, label='Safety threshold')
ax2.axvline(forecast['days_until_stockout'], color='#c0392b',
            linestyle=':', linewidth=2, label=f"Stockout: Day {forecast['days_until_stockout']:.1f}")
ax2.set_xlabel('Days from now'); ax2.set_ylabel('Stock remaining (units)')
ax2.set_title('Stock Depletion Trajectory', fontweight='bold')
ax2.legend(fontsize=9); ax2.set_xlim(0, 14); ax2.set_ylim(0)

# 3. Distributor confidence bars
ax3 = fig.add_subplot(gs[2])
names  = [r['distributor'].replace(' ', '\n') for r in ranked]
confs  = [r['confidence'] for r in ranked]
colors3 = ['#e74c3c', '#95a5a6', '#bdc3c7']
bars3  = ax3.bar(names, confs, color=colors3, edgecolor='white', width=0.5)
ax3.set_ylabel('Recommendation Confidence')
ax3.set_title('Distributor Rankings', fontweight='bold')
ax3.set_ylim(0, 1.1)
for bar, val in zip(bars3, confs):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.1%}', ha='center', fontsize=11, fontweight='bold')
ax3.tick_params(axis='x', labelsize=9)

fig.suptitle(f"GraminRoute — Ramesh's Shop ({RAMESH['village']}) · Full Pipeline Summary",
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig(MODELS_DIR / 'demo_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()


## ✅ Training Complete

All models have been saved to `backend/models/`:

| File | Description |
|------|-------------|
| `xgb_risk_model.pkl` | XGBoost Risk Model (9 features → risk score) |
| `xgb_recommender.pkl` | XGBoost Recommender (5 context features → distributor rank) |
| `spatial_gnn.pth` | SpatialGNN weights (GATv2, in_dim=10) |

### Run the API
```bash
cd backend
uvicorn api.main:app --reload
# → http://localhost:8000/docs
```

### Test the pipeline
```bash
curl -X POST http://localhost:8000/recommend_distributor \
  -H "Content-Type: application/json" \
  -d '{"shop_id":"GHPR-042","lat":17.9,"lon":79.3,"current_stock":18,"daily_sales":8,"product_name":"Rice (50kg)"}'
```
